# IntrinsicZernikes calib tables — OCS map + per-detector CCS viewer

Plots the source tables produced by `bin.src/run_make_calib_tables.py`:
the single per-instrument **OCS** map (`intrinsic_aberrations_OCS.parquet`)
and one detector's **CCS** table
(`intrinsic_aberrations_CCS_det<NNN>.parquet` = smooth camera field +
that CCD's height Z4 piston). Pure numpy/matplotlib/astropy.


## 1. Parameters

In [ ]:
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

# Directory written by run_make_calib_tables.py.
tables_dir = "/sdf/group/rubin/repo/aos_imsim/gmegias/intrinsic_zernikes/v2"
detector = 90            # which detector's CCS table to view

zernikes = "all"        # "all", or a list of Noll indices
fp_radius_deg = 1.75
pct = 98

## 2. Read the OCS map and the detector's CCS table

In [ ]:
def read_table(path):
    t = Table.read(str(path), format="parquet")
    x = t["x"].to("deg").value; y = t["y"].to("deg").value
    js = sorted(int(m.group(1)) for c in t.colnames
                for m in [re.match(r"Z(\d+)$", c)] if m)
    return x, y, js, {j: t[f"Z{j}"].to("um").value for j in js}, dict(t.meta)

xo, yo, js_o, ocs, ocs_meta = read_table(Path(tables_dir) / "intrinsic_aberrations_OCS.parquet")
xc, yc, js_c, ccs, ccs_meta = read_table(
    Path(tables_dir) / f"intrinsic_aberrations_CCS_det{detector:03d}.parquet")
js = sorted(set(js_o) & set(js_c))
if zernikes != "all":
    js = [j for j in js if j in zernikes]
ocs_only = set(int(j) for j in (ocs_meta.get("ocs_only") or []))
print(f"OCS {len(xo)} pts; CCS(det {detector}) {len(xc)} pts; "
      f"piston_z4={ccs_meta.get('piston_z4_um')} um")
print(f"Zernikes: {js}")

## 3. Helper: focal-plane map

In [ ]:
def plot_map(ax, x, y, vals, title, vlim, cmap="RdBu_r"):
    v = np.asarray(vals, dtype=float); fin = np.isfinite(v)
    tcf = ax.tricontourf(x[fin], y[fin], v[fin],
                         levels=np.linspace(-vlim, vlim, 21), cmap=cmap, extend="both")
    ax.add_patch(plt.Circle((0, 0), fp_radius_deg, fill=False, ec="k", lw=0.6, alpha=0.4))
    ax.set_aspect("equal")
    ax.set_xlim(-fp_radius_deg, fp_radius_deg); ax.set_ylim(-fp_radius_deg, fp_radius_deg)
    ax.set_title(title, fontsize=9); ax.set_xlabel("x [deg]"); ax.set_ylabel("y [deg]")
    return tcf


def vlim_for(*arrays):
    vv = np.concatenate([np.asarray(a, float)[np.isfinite(a)] for a in arrays])
    return max(float(np.nanpercentile(np.abs(vv), pct)), 1e-6) if vv.size else 1.0

## 4. Plot OCS (telescope) and CCS (camera+heights)
One row per Zernike; shared color scale. Higher-order j are OCS-only, so their CCS is ~0 (Z4 also carries the per-CCD height piston).

In [ ]:
for j in js:
    O = ocs[j]; C = ccs[j]
    vlim = vlim_for(O, C)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.6), layout="constrained")
    tcf = plot_map(axs[0], xo, yo, O, f"Z{j}  OCS (telescope)", vlim)
    note = "   [OCS-only: C\u22480]" if j in ocs_only else ""
    plot_map(axs[1], xc, yc, C, f"Z{j}  CCS det{detector} (camera){note}", vlim)
    fig.colorbar(tcf, ax=axs, shrink=0.85, label="\u00b5m")
    fig.suptitle(f"IntrinsicZernikes calib \u2014 Z{j}", fontsize=12)
    plt.show()
    plt.close(fig)